In [ ]:
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


def get_data_from_kaggle():
    from kagglehub.datasets import dataset_download
    downloaded_path = dataset_download(
        "blastchar/telco-customer-churn",
        path="WA_Fn-UseC_-Telco-Customer-Churn.csv",
        output_dir=".",
    )
    os.rename(downloaded_path, "./telco.csv")

In [ ]:
# ==================== 0. Check if the dataset exists, if not download it from Kaggle ====================
if not os.path.exists(r"./telco.csv"):
    print("download")
    get_data_from_kaggle()

assert os.path.exists(r"./telco.csv"), "Dataset not found. Please download it from Kaggle."

In [ ]:
# ==================== 1. Load and inspect the data ====================
data = pd.read_csv(r"./telco.csv")
data = data.convert_dtypes()

# print(data.head())
# data.info()

# ---------- 1.1 change `No internet service` to `No`.
data.replace({ "No internet service": "No", "No phone service": "No"}, inplace=True)

# ---------- 1.2 Convert Yes/No and Male/Female to boolean(integer).
for col in data.columns:
    st = set(data[col].unique())
    if st == {"Yes", "No"}: data[col] = data[col].map({"Yes": 1, "No": 0})
    elif st == {"Male", "Female"}: data[col] = data[col].map({"Female": 0, "Male": 1})


data.info()

In [ ]:
# ==================== 2. Define the variables ====================
# ---------- 2.1 One-hot encode the categorical variables.
data = pd.get_dummies(data, columns=["InternetService"], drop_first=True)
bool_cols = data.select_dtypes(include=["boolean"]).columns
data[bool_cols] = data[bool_cols].astype("Int64")

# ---------- 2.2 Define the features and target variable.
service_cols = [
    "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "InternetService_Fiber optic", "InternetService_No"
]
features = data[service_cols]
target = data["MonthlyCharges"]

# features.info()
# target.info()

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(features, target, train_size=0.8, random_state=818)


# ==================== 4. Fit a multivariable linear regression ====================
model = LinearRegression()
model.fit(X_train, y_train)


# ==================== 5. Report the parameters ====================
# print("Coefficients:", model.coef_)
# print("Intercept:", model.intercept_)

b = model.intercept_  # base plan charge
coefficient_list = (pd.DataFrame({'service': service_cols, 'cost': model.coef_.round(2)})
                    .sort_values('cost', ascending=False)
                    .reset_index(drop=True))  # dollar contribution of each service

print(f"Base monthly charge (intercept): ${b:.2f}")
print(coefficient_list)

In [ ]:
# ==================== 6. Evaluate on the test set ====================
mae = mean_absolute_error(y_test, model.predict(X_test))
mse = mean_squared_error(y_test, model.predict(X_test))
rmse = np.sqrt(mse)
r2 = r2_score(y_test, model.predict(X_test))
print(f"Mean Absolute Error: {mae:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")
print(f"R-squared: {r2:.2f}")

In [ ]:
# ==================== 7. Compare to a baseline ====================
# ---------- 7.1 Data preparation
addon_cols = ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
data_new = pd.DataFrame({
    "num_addons": data[addon_cols].sum(axis=1),  # sum across the row = how many add-ons
    "MonthlyCharges": data["MonthlyCharges"],
})

# ---------- 7.2 Split the data and train a baseline model
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(data_new[["num_addons"]], data_new["MonthlyCharges"], train_size=0.8, random_state=818)
base_model = LinearRegression()
base_model.fit(X_train_new, y_train_new)

# ---------- 7.3 Evaluate the baseline model
mae_base = mean_absolute_error(y_test_new, base_model.predict(X_test_new))
mse_base = mean_squared_error(y_test_new, base_model.predict(X_test_new))
rmse_base = np.sqrt(mse_base)
r2_base = r2_score(y_test_new, base_model.predict(X_test_new))
print(f"Baseline Mean Absolute Error: {mae_base:.2f}")
print(f"Baseline Root Mean Squared Error: {rmse_base:.2f}")
print(f"Baseline R-squared: {r2_base:.2f}")

In [ ]:
# ==================== 8. Interpret the results ====================
coef_table = pd.Series(model.coef_, index=service_cols).sort_values(ascending=False)
print(f"Base monthly charge (intercept): ${model.intercept_:.2f}")
print("Estimated $/month contribution of each service:")
for name, coef in coef_table.items(): print(f"  {name}: {coef:+.2f} $/month")

# higher R2 is better. (Multi Variable Model R2 is higher, so it's better)
# lower MAE & RMSE is better. (Multi Variable Model MAE and RMSE is lower, so it's better)

In [ ]:
# ==================== 9. Predict a new customer ====================
new_customer = pd.DataFrame([{
    "PhoneService": 1,
    "MultipleLines": 1,
    "OnlineSecurity": 1,
    "OnlineBackup": 0,
    "DeviceProtection": 0,
    "TechSupport": 1,
    "StreamingTV": 1,
    "StreamingMovies": 0,
    "InternetService_Fiber optic": 1,
    "InternetService_No": 0,
}], columns=service_cols)

predicted_charge = model.predict(new_customer)[0]
print(f"Predicted monthly charge for the new customer: ${predicted_charge:.2f}")

Base monthly charge (intercept): $24.95
                       service   cost
0  InternetService_Fiber optic  24.93
1                 PhoneService  20.08
2              StreamingMovies   9.99
3                  StreamingTV   9.95
4                  TechSupport   5.06
5               OnlineSecurity   5.02
6                MultipleLines   5.01
7             DeviceProtection   5.01
8                 OnlineBackup   4.97
9           InternetService_No -25.07

# Questions:

Q1: What does the base plan cost before any add-ons? (the intercept)
A1: The base plan cost before any add-ons is $24.95.

Q2: How much does each individual add-on contribute to the monthly bill?
A2: The contribution of each individual add-on to the monthly bill is as follows:
    - InternetService_Fiber optic: $24.93
    - PhoneService: $20.08
    - StreamingMovies: $9.99
    - StreamingTV: $9.95
    - TechSupport: $5.06
    - OnlineSecurity: $5.02
    - MultipleLines: $5.01
    - DeviceProtection: $5.01
    - OnlineBackup: $4.97 
    - InternetService_No: -$25.07

Q3: Which add-on is the most expensive, and which is the cheapest?
A3: The most expensive add-on is InternetService_Fiber optic at $24.93, and the cheapest add-on is OnlineBackup at $4.97.

Q4: If a customer adds fiber internet plus both streaming services, how much should their bill rise?
A4: The bill should be rised $24.93 (Fiber optic) + $9.99 (StreamingMovies) + $9.95 (StreamingTV) = $44.87. 

Q5: Given a specific bundle (say, phone service , multiple lines, and tech support), what monthly charge
should we expect?
A5: The expected monthly charge is $24.95 (base) + $20.08 (PhoneService) + $5.01 (MultipleLines) + $5.06 (TechSupport) = $55.10.

Q6: Is knowing which services a customer has meaningfully better than just knowing how many they
have, for predicting the bill?
A6: Yes, knowing which specific services a customer has is meaningfully better than just knowing how many they have for predicting the bill. We can see this by comparing the metrics of the multi-variable model and the baseline model.

Q7: How accurate are our predictions on customers the model hasn't seen within a few dollars, or off by
more?
A7: Within a few dollars, the model's predictions are fairly accurate. The MAE and RMSE values indicate that the model's predictions are generally close to the actual monthly charges.

Q8: Do the model's learned prices match Telco's actual price list, and are any coefficients surprising
(negative, or larger/smaller than expected)?
A8: Yes, the model's learned prices generally match Telco's actual price list. The negative coefficient for InternetService_No is normal since it indicates that not having internet service reduces the monthly charge.